# CV4CHL - TCN Item 14: Per-Item 1D-TCN on Phase-Normalized Gait Cycles

**Platform**: Colab T4 GPU
**Runtime**: GPU required (~30-60 min full run, ~2 min dry run)
**Goal**: Add temporal sequence modeling for EVGS item 14

---

## Approach

| Aspect | Patient-level XGB | This (tcn_item14) |
|--------|----------------------|---------------|
| Input | 63 handcrafted features per patient | 4-channel angle sequence per gait cycle |
| Granularity | Patient-level (94 train samples) | Cycle-level (~2800 train samples) |
| Model | XGBoost / LightGBM / CatBoost | 1D-TCN (~3K params per item) |
| Prediction | Direct per-patient binary | Cycle probs -> patient aggregation |

## Design rationale

Item 14 (Pelvic Obliquity) has temporal dynamics within a gait cycle that
patient-level handcrafted features cannot capture. A 1D-TCN operating on
phase-normalized gait cycles addresses this with:

- **phase-normalized input**: each cycle resampled to a fixed length so the
  model sees comparable timing across patients;
- **gait-cycle-level training (~2800 samples)**: orders of magnitude more
  training instances than the 94 patient-level samples;
- **ultra-lightweight (~3-5K params per item)**: small enough that the
  cycle-level sample count is sufficient to fit without overfitting.


In [ ]:
# === Cell 1: Setup & Imports ===
import os
import sys
import time
import json
import pickle
import warnings
from datetime import datetime
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)

# --- Timing helpers ---
NOTEBOOK_NAME = 'CV4CHL_07_train_tcn_item14'
_cell_times = {}
_cell_start = None

def cell_start(name):
    global _cell_start
    _cell_start = time.time()
    _cell_times[name] = None
    print(f'\u25b6 {name}')

def cell_end(name):
    elapsed = time.time() - _cell_start
    _cell_times[name] = elapsed
    print(f'\u2713 {name} \u2014 {elapsed:.1f}s ({elapsed/60:.1f}min)')

# --- PyTorch ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- GPU detection ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# GPU stats tracking
_gpu_stats = {}
def log_gpu_stats(step_name=''):
    if not torch.cuda.is_available():
        return
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    util_pct = allocated / total * 100
    _gpu_stats[step_name] = {
        'allocated_gb': allocated, 'reserved_gb': reserved,
        'total_gb': total, 'util_pct': util_pct,
    }
    return allocated, reserved, total, util_pct

print(f'\nNotebook: {NOTEBOOK_NAME}')
print(f'Started:  {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'numpy={np.__version__}  pandas={pd.__version__}  torch={torch.__version__}')
print(f'GPU: {gpu_name} | VRAM: {vram_gb:.0f} GB | Device: {DEVICE}')
if not torch.cuda.is_available():
    print('\u26a0\ufe0f WARNING: No GPU detected. Training will be VERY slow on CPU.')


In [ ]:
# === Cell 2: Config & Load Data ===
cell_start('Cell 2: Config & Load Data')

# ============================
# DRY_RUN toggle
# ============================
DRY_RUN = os.environ.get('CV4CHL_DRY_RUN', '0') == '1'  # set CV4CHL_DRY_RUN=1 for ~2-min smoke test

if DRY_RUN:
    DRY_ITEMS = [3, 9, 11]     # Only train 3 items
    DRY_FOLDS = 5              # Only 5 LOSO folds per item
    DRY_EPOCHS = 5             # Fewer epochs
    print('DRY RUN: 3 items, 5 folds, 5 epochs (~2 min)')
else:
    print('FULL RUN: 17 items, 94 folds, 30 epochs (~30-60 min)')

# --- Environment detection ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        '/content/drive/MyDrive/CVPR_CH4CHL',
    )
else:
    ROOT = os.environ.get('CV4CHL_REPO_ROOT', os.getcwd())

PROC_DIR   = f'{ROOT}/1_data/processed'
SUBMIT_DIR = f'{ROOT}/5_outputs/submissions'
OUTPUT_DIR    = f'{SUBMIT_DIR}/tcn_item14'                       # canonical artifact only (Cell 8b)
DIAGNOSTIC_DIR = f'{ROOT}/5_outputs/diagnostics/tcn_item14'      # per-item / combo / variant candidates (Cell 8)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DIAGNOSTIC_DIR, exist_ok=True)
print(f'ROOT      : {ROOT}')
print(f'PROC_DIR  : {PROC_DIR}')
print(f'OUTPUT_DIR   : {OUTPUT_DIR}')

# --- Constants ---
T1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]

# Track-2 rows are preserved from the current base CSV;
# Track-2 predictions are generated only by the Track-2 clinical notebook.

LABEL_COLS = [f'evgs_{i}' for i in range(1, 18)]

KP = {
    'nose': 0,
    'left_shoulder': 5, 'right_shoulder': 6,
    'left_elbow': 7, 'right_elbow': 8,
    'left_wrist': 9, 'right_wrist': 10,
    'left_hip': 11, 'right_hip': 12,
    'left_knee': 13, 'right_knee': 14,
    'left_ankle': 15, 'right_ankle': 16,
    'left_big_toe': 17, 'left_small_toe': 18, 'left_heel': 19,
    'right_big_toe': 20, 'right_small_toe': 21, 'right_heel': 22,
}

# TCN hyperparameters
SEQ_LEN = 64          # Phase-normalized cycle length
N_CHANNELS = 4        # knee, ankle, hip, trunk
TCN_HIDDEN = 32       # Hidden channels per TCN layer
TCN_LAYERS = 3        # Number of TCN blocks (dilation 1,2,4)
TCN_DROPOUT = 0.3     # Dropout rate
LR = 1e-3             # Learning rate
WEIGHT_DECAY = 1e-4   # L2 regularization
N_EPOCHS = 30 if not DRY_RUN else DRY_EPOCHS
PATIENCE = 5          # Early stopping patience
BATCH_SIZE = 64       # Fits any GPU

# Butterworth filter params (matched to the preprocessing notebook)
FS = 30.0
CUTOFF = 6.0
FILTER_ORDER = 4

# Gait cycle params (SAME as phase-feature preprocessing)
MIN_CYCLE_FRAMES = 20
MAX_CYCLE_FRAMES = 120
VALID_STANCE_LO = 0.35
VALID_STANCE_HI = 0.85

# --- Load keypoints.pkl ---
kp_path = f'{PROC_DIR}/keypoints.pkl'
assert os.path.exists(kp_path), f'Missing: {kp_path}'
with open(kp_path, 'rb') as f:
    kp_raw = pickle.load(f)
print(f'\nLoaded keypoints.pkl: {len(kp_raw)} video entries')

sample_key = next(iter(kp_raw))
if isinstance(sample_key, tuple):
    _kp_by_pid = defaultdict(dict)
    for (pid, vname), vdata in kp_raw.items():
        _kp_by_pid[pid][vname] = vdata
    kp_data = dict(_kp_by_pid)
else:
    kp_data = kp_raw
print(f'Patients with keypoints: {len(kp_data)}')

# --- Load train_ready.pkl ---
tr_path = f'{PROC_DIR}/train_ready.pkl'
assert os.path.exists(tr_path), f'Missing: {tr_path}'
with open(tr_path, 'rb') as f:
    tr_data = pickle.load(f)
df_train = tr_data['track1_train'].copy()
df_test  = tr_data['track1_test'].copy()
print(f'Train labels: {df_train.shape} | Test rows: {df_test.shape}')
print(f'Train unique pids: {df_train["patient_id"].nunique()}')

# Build label lookup: (pid, side) -> {item_i: 0 or 1}
label_lookup = {}
for _, row in df_train.iterrows():
    pid = int(row['patient_id'])
    side = row['side']
    labels = {i: int(row[f'evgs_{i}']) for i in range(1, 18)}
    label_lookup[(pid, side)] = labels

train_pids = sorted(df_train['patient_id'].unique())
test_pids = sorted(df_test['patient_id'].unique())
all_pids = sorted(set(train_pids) | set(test_pids))
print(f'Train PIDs: {len(train_pids)} | Test PIDs: {len(test_pids)}')

# --- Load base submission ---
BASE_PATH = f'{SUBMIT_DIR}/reference_baseline.csv'
if not os.path.exists(BASE_PATH):
    raise FileNotFoundError(
        f'Base submission not found: {BASE_PATH}. '
        'Either run `python reproduce.py` from a clean state (which stages '
        'submissions/reference/xgb_baseline.csv into the runtime path), or '
        'run notebook 02 (CV4CHL_02_train_xgb_baseline.ipynb) first.'
    )
df_base = pd.read_csv(BASE_PATH)
base_path_used = BASE_PATH
print(f'\nBase submission loaded: {BASE_PATH}  shape={df_base.shape}')

cell_end('Cell 2: Config & Load Data')

In [ ]:
# === Cell 3: Gait Cycle Extraction + Phase Normalization ===
cell_start('Cell 3: Gait Cycle Extraction')

# ---------------------------------------------------------------
# Signal processing helpers (identical to phase-feature preprocessing/evgs_rules)
# ---------------------------------------------------------------

def butter_lowpass(x, cutoff=CUTOFF, fs=FS, order=FILTER_ORDER):
    """Butterworth low-pass filter (same as phase-feature preprocessing)."""
    x = np.asarray(x, dtype=float)
    if len(x) < 3 * order + 1:
        return x.copy()
    nyq = 0.5 * fs
    b, a = butter(order, min(cutoff / nyq, 0.99), btype='low')
    try:
        return filtfilt(b, a, x, padtype='odd',
                        padlen=min(3 * max(len(b), len(a)), len(x) - 1))
    except Exception:
        return x.copy()


def smooth_kps(kps_2d):
    """Smooth (T, 2) keypoint trajectory with Butterworth."""
    kps_2d = np.asarray(kps_2d, dtype=float)
    if kps_2d.ndim != 2 or len(kps_2d) < 13:
        return kps_2d.copy()
    out = kps_2d.copy()
    out[:, 0] = butter_lowpass(kps_2d[:, 0])
    out[:, 1] = butter_lowpass(kps_2d[:, 1])
    return out


def smooth_all_keypoints(kps):
    """Smooth (T, 23, 2) keypoints array."""
    T, n_kp, _ = kps.shape
    kps_s = kps.copy()
    if T < 13:
        return kps_s
    for j in range(n_kp):
        kps_s[:, j, 0] = butter_lowpass(kps[:, j, 0])
        kps_s[:, j, 1] = butter_lowpass(kps[:, j, 1])
    return kps_s


# ---------------------------------------------------------------
# Gait event detection (Zeni method, same as phase-feature preprocessing)
# ---------------------------------------------------------------

def detect_gait_events_zeni(kps_smooth, side='left', fps=FS):
    """Zeni method: heel strikes = maxima of (heel_x - sacrum_x).

    Handles bidirectional walking by auto-detecting walk direction.
    Returns dict with 'hs_frames', 'to_frames', 'valid'.
    """
    T = len(kps_smooth)
    result = {'hs_frames': np.array([], dtype=int),
              'to_frames': np.array([], dtype=int), 'valid': False}
    if T < 30:
        return result

    sacrum = (kps_smooth[:, KP['left_hip']] + kps_smooth[:, KP['right_hip']]) / 2.0
    heel = kps_smooth[:, KP[f'{side}_heel']]
    toe = kps_smooth[:, KP[f'{side}_big_toe']]

    # Detect walking direction
    slope = np.polyfit(np.arange(T, dtype=float), sacrum[:, 0], 1)[0]
    direction = 1.0 if slope >= 0 else -1.0

    # Relative positions in walking direction
    hs_signal = direction * (heel[:, 0] - sacrum[:, 0])
    to_signal = direction * (toe[:, 0] - sacrum[:, 0])

    signal_range = np.ptp(hs_signal)
    if signal_range < 5.0:
        return result

    min_dist = max(int(fps * 0.4), 10)
    hs_prom = max(signal_range * 0.15, 3.0)
    to_range = np.ptp(to_signal)
    to_prom = max(to_range * 0.15, 3.0)

    hs_frames, _ = find_peaks(hs_signal, distance=min_dist, prominence=hs_prom)
    to_frames, _ = find_peaks(-to_signal, distance=min_dist, prominence=to_prom)

    result['hs_frames'] = hs_frames
    result['to_frames'] = to_frames
    if len(hs_frames) >= 2 and len(to_frames) >= 1:
        result['valid'] = True
    return result


def extract_gait_cycles(events, min_frames=MIN_CYCLE_FRAMES, max_frames=MAX_CYCLE_FRAMES):
    """Extract valid gait cycles from detected events (same as phase-feature preprocessing)."""
    cycles = []
    hs = events['hs_frames']
    to = events['to_frames']
    if len(hs) < 2:
        return cycles

    for i in range(len(hs) - 1):
        hs_start = int(hs[i])
        hs_end = int(hs[i + 1])
        cycle_len = hs_end - hs_start
        if cycle_len < min_frames or cycle_len > max_frames:
            continue

        to_cand = to[(to > hs_start) & (to < hs_end)]
        if len(to_cand) == 0:
            continue

        to_frame = int(to_cand[0])
        stance_ratio = (to_frame - hs_start) / cycle_len
        if not (VALID_STANCE_LO <= stance_ratio <= VALID_STANCE_HI):
            continue

        cycles.append({
            'hs_start': hs_start,
            'hs_end': hs_end,
            'to_frame': to_frame,
            'stance_start': hs_start,
            'stance_end': to_frame,
            'swing_start': to_frame,
            'swing_end': hs_end,
            'cycle_length': cycle_len,
            'stance_ratio': stance_ratio,
        })
    return cycles


# ---------------------------------------------------------------
# Signed joint angle computation (same as evgs_rules)
# ---------------------------------------------------------------

def signed_angle_3pt(a, b, c):
    """Signed angle at vertex b, formed by a-b-c. Returns degrees.
    Uses atan2 for unambiguous signed angles.
    """
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    angle = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))
    cross = ba[0] * bc[1] - ba[1] * bc[0]
    return float(angle if cross >= 0 else -angle)


def compute_4ch_angles(kps_smooth, side='left'):
    """Compute 4-channel angle time series for one side.

    Channels:
      0: knee flexion (hip-knee-ankle)
      1: ankle dorsiflexion (knee-ankle-toe)
      2: hip flexion (shoulder-hip-knee)
      3: trunk lean (angle of trunk from vertical)

    Returns: (T, 4) float32 array
    """
    T = len(kps_smooth)
    angles = np.zeros((T, 4), dtype=np.float32)

    hip_idx = KP[f'{side}_hip']
    knee_idx = KP[f'{side}_knee']
    ankle_idx = KP[f'{side}_ankle']
    toe_idx = KP[f'{side}_big_toe']
    shoulder_idx = KP[f'{side}_shoulder']

    for t in range(T):
        k = kps_smooth[t]

        # Channel 0: Knee flexion (hip-knee-ankle)
        angles[t, 0] = signed_angle_3pt(k[hip_idx], k[knee_idx], k[ankle_idx])

        # Channel 1: Ankle angle (knee-ankle-toe)
        angles[t, 1] = signed_angle_3pt(k[knee_idx], k[ankle_idx], k[toe_idx])

        # Channel 2: Hip flexion (shoulder-hip-knee)
        angles[t, 2] = signed_angle_3pt(k[shoulder_idx], k[hip_idx], k[knee_idx])

        # Channel 3: Trunk lean from vertical
        ms = (k[KP['left_shoulder']] + k[KP['right_shoulder']]) / 2
        mh = (k[KP['left_hip']] + k[KP['right_hip']]) / 2
        trunk_vec = ms - mh  # shoulder - hip (upward in real world)
        vertical = np.array([0, -1], dtype=np.float32)  # up in image coords (y-down)
        cos_a = np.dot(trunk_vec, vertical) / (np.linalg.norm(trunk_vec) + 1e-8)
        angles[t, 3] = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))

    return angles


def phase_normalize(cycle_angles, target_len=SEQ_LEN):
    """Resample (cycle_len, n_ch) to (target_len, n_ch) via linear interpolation."""
    cycle_len, n_ch = cycle_angles.shape
    if cycle_len < 2:
        return np.zeros((target_len, n_ch), dtype=np.float32)
    x_old = np.linspace(0, 1, cycle_len)
    x_new = np.linspace(0, 1, target_len)
    out = np.zeros((target_len, n_ch), dtype=np.float32)
    for ch in range(n_ch):
        out[:, ch] = np.interp(x_new, x_old, cycle_angles[:, ch])
    return out


# ---------------------------------------------------------------
# Video helpers
# ---------------------------------------------------------------

def get_video_view(vdata, vname):
    """Determine video view from metadata or filename."""
    v = vdata.get('view') if isinstance(vdata, dict) else None
    if isinstance(v, str) and v:
        return v.lower()
    name = vname.lower() if isinstance(vname, str) else ''
    for tag in ('forward', 'backward', 'left', 'right'):
        if f'_{tag}_' in name:
            return tag
    return ''


def get_video_arrays(vdata):
    """Extract keypoints and scores from video data dict."""
    kps = vdata.get('keypoints', vdata.get('kps'))
    sc = vdata.get('scores')
    if kps is None or sc is None:
        return None, None
    kps = np.asarray(kps, dtype=float)
    sc = np.asarray(sc, dtype=float)
    if kps.ndim != 3 or kps.shape[1] < 23:
        return None, None
    return kps[:, :23, :2], sc[:, :23]


# ---------------------------------------------------------------
# Main extraction loop
# ---------------------------------------------------------------

all_cycles = []
n_videos_used = 0
n_videos_skipped = 0
n_cycles_total = 0
SCORE_THRESH = 0.4
MIN_VALID_FRAMES = 30

for pid in tqdm(all_pids, desc='Extracting gait cycles'):
    if pid not in kp_data:
        continue

    videos = kp_data[pid]
    for vname, vdata in videos.items():
        view = get_video_view(vdata, vname)

        # Only use sagittal videos (left/right view)
        if view not in ('left', 'right'):
            n_videos_skipped += 1
            continue

        kps, scores = get_video_arrays(vdata)
        if kps is None or len(kps) < MIN_VALID_FRAMES:
            n_videos_skipped += 1
            continue

        # Check keypoint quality
        side = view  # left view -> left side, right view -> right side
        needed_kps = [KP[f'{side}_hip'], KP[f'{side}_knee'], KP[f'{side}_ankle'],
                      KP[f'{side}_heel'], KP[f'{side}_big_toe'], KP[f'{side}_shoulder'],
                      KP['left_hip'], KP['right_hip'],
                      KP['left_shoulder'], KP['right_shoulder']]
        valid_mask = np.ones(len(kps), dtype=bool)
        for idx in needed_kps:
            valid_mask &= (scores[:, idx] >= SCORE_THRESH)
        if valid_mask.sum() < MIN_VALID_FRAMES:
            n_videos_skipped += 1
            continue

        # Use only frames with valid keypoints
        kps_valid = kps[valid_mask]
        kps_smooth = smooth_all_keypoints(kps_valid)

        # Detect gait events
        events = detect_gait_events_zeni(kps_smooth, side=side)
        if not events['valid']:
            n_videos_skipped += 1
            continue

        # Extract cycles
        cycles = extract_gait_cycles(events)
        if not cycles:
            n_videos_skipped += 1
            continue

        # Compute angle time series for entire video
        angles_full = compute_4ch_angles(kps_smooth, side=side)

        # Extract and phase-normalize each cycle
        for ci, cyc in enumerate(cycles):
            s, e = cyc['hs_start'], cyc['hs_end']
            cycle_angles = angles_full[s:e]
            if len(cycle_angles) < MIN_CYCLE_FRAMES:
                continue

            # Phase-normalize to SEQ_LEN frames
            norm_angles = phase_normalize(cycle_angles, SEQ_LEN)

            # Get labels (None for test patients)
            labels = label_lookup.get((pid, side), None)

            all_cycles.append({
                'patient_id': pid,
                'side': side,
                'video_name': vname,
                'cycle_idx': ci,
                'angles': norm_angles,       # (64, 4)
                'stance_ratio': cyc['stance_ratio'],
                'cycle_length': cyc['cycle_length'],
                'labels': labels,            # dict {1: 0/1, ..., 17: 0/1} or None
                'is_train': pid in train_pids,
            })
            n_cycles_total += 1

        n_videos_used += 1

# Split into train/test
train_cycles = [c for c in all_cycles if c['is_train']]
test_cycles = [c for c in all_cycles if not c['is_train']]

# Collect patient IDs that have cycles
train_pids_with_cycles = sorted(set(c['patient_id'] for c in train_cycles))
test_pids_with_cycles = sorted(set(c['patient_id'] for c in test_cycles))

print(f'\nGait cycle extraction complete:')
print(f'  Videos used: {n_videos_used}')
print(f'  Videos skipped: {n_videos_skipped}')
print(f'  Total cycles: {n_cycles_total}')
print(f'  Train cycles: {len(train_cycles)} from {len(train_pids_with_cycles)} patients')
print(f'  Test cycles: {len(test_cycles)} from {len(test_pids_with_cycles)} patients')

# Quick stats
cycles_per_patient = Counter(c['patient_id'] for c in train_cycles)
if cycles_per_patient:
    print(f'\nCycles per train patient: mean={np.mean(list(cycles_per_patient.values())):.1f}, '
          f'median={np.median(list(cycles_per_patient.values())):.0f}, '
          f'min={min(cycles_per_patient.values())}, max={max(cycles_per_patient.values())}')

# Check which train patients have no cycles
missing_pids = set(train_pids) - set(train_pids_with_cycles)
if missing_pids:
    print(f'\n\u26a0\ufe0f Train patients with NO cycles (will use majority vote): {sorted(missing_pids)}')

cell_end('Cell 3: Gait Cycle Extraction')


In [ ]:
# === Cell 4: Data Augmentation ===
cell_start('Cell 4: Data Augmentation')

def augment_speed_change(angles, speed_factor):
    """Linear speed change: resample cycle at different speed.
    speed_factor=0.9 -> 10% slower, speed_factor=1.1 -> 10% faster.
    """
    T, n_ch = angles.shape
    n_src = int(T * speed_factor)
    n_src = max(4, min(n_src, T * 3))
    x_old = np.linspace(0, 1, T)
    x_new = np.linspace(0, 1, n_src)
    tmp = np.zeros((n_src, n_ch), dtype=np.float32)
    for ch in range(n_ch):
        tmp[:, ch] = np.interp(x_new, x_old, angles[:, ch])
    out = np.zeros_like(angles)
    x_tmp = np.linspace(0, 1, n_src)
    x_out = np.linspace(0, 1, T)
    for ch in range(n_ch):
        out[:, ch] = np.interp(x_out, x_tmp, tmp[:, ch])
    return out


def augment_temporal_warp(angles, factor):
    """Non-linear temporal warping using power function."""
    T, n_ch = angles.shape
    x_old = np.linspace(0, 1, T)
    x_warped = np.linspace(0, 1, T) ** factor
    out = np.zeros_like(angles)
    for ch in range(n_ch):
        out[:, ch] = np.interp(x_warped, x_old, angles[:, ch])
    return out


def augment_noise(angles, std_deg=1.5):
    """Add Gaussian noise to angle channels."""
    noise = np.random.randn(*angles.shape).astype(np.float32) * std_deg
    return angles + noise


def build_augmented_dataset(cycles, item_i, do_augment=True):
    """Build training arrays for a specific item with augmentation.

    Returns:
        X: (N, 4, SEQ_LEN) float32 - channels first for Conv1d
        y: (N,) int64 - binary labels
        pids: (N,) int64 - patient IDs (for LOSO grouping)
    """
    X_list = []
    y_list = []
    pid_list = []

    for cyc in cycles:
        if cyc['labels'] is None:
            continue
        label = cyc['labels'].get(item_i)
        if label is None:
            continue

        angles = cyc['angles']  # (SEQ_LEN, 4)
        pid = cyc['patient_id']

        # Original sample
        X_list.append(angles.T)  # (4, SEQ_LEN)
        y_list.append(label)
        pid_list.append(pid)

        if do_augment:
            # Speed changes: 0.9x and 1.1x
            for speed in [0.9, 1.1]:
                aug = augment_speed_change(angles, speed)
                X_list.append(aug.T)
                y_list.append(label)
                pid_list.append(pid)

            # Gaussian noise: 2 versions
            for _ in range(2):
                aug = augment_noise(angles, std_deg=1.5)
                X_list.append(aug.T)
                y_list.append(label)
                pid_list.append(pid)

            # Temporal warp: mild non-linear warping
            for wf in [0.8, 1.2]:
                aug = augment_temporal_warp(angles, wf)
                X_list.append(aug.T)
                y_list.append(label)
                pid_list.append(pid)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    pids = np.array(pid_list, dtype=np.int64)
    return X, y, pids


# Test augmentation on a sample
if train_cycles:
    sample = train_cycles[0]
    sample_aug = augment_speed_change(sample['angles'], 0.9)
    sample_noise = augment_noise(sample['angles'])
    print(f'Original shape: {sample["angles"].shape}')
    print(f'Speed-changed shape: {sample_aug.shape}')
    print(f'Noised shape: {sample_noise.shape}')

# Count augmentation multiplier: 1 original + 2 speed + 2 noise + 2 warp = 7x
AUG_FACTOR = 7
print(f'\nAugmentation factor: {AUG_FACTOR}x')
print(f'Expected augmented train samples per item: ~{len(train_cycles) * AUG_FACTOR}')

cell_end('Cell 4: Data Augmentation')


In [ ]:
# === Cell 5: 1D-TCN Model Definition ===
cell_start('Cell 5: Model Definition')


class TCNBlock(nn.Module):
    """Single TCN block: Conv1d -> BatchNorm -> ReLU -> Dropout + residual."""

    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.3):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              padding=padding, dilation=dilation)
        self.bn = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        out = self.bn(out)
        out = F.relu(out)
        out = self.dropout(out)
        return out + self.residual(x)


class TCN1D(nn.Module):
    """Lightweight 1D Temporal Convolutional Network for per-item EVGS prediction.

    Architecture: N TCN blocks (with residual connections) + GlobalAvgPool + FC head
    Dilation: [1, 2, 4, ...] for exponential receptive field growth

    Input: (batch, channels=4, seq_len=64)
    Output: (batch,) logits (use BCEWithLogitsLoss)
    """

    def __init__(self, in_channels=N_CHANNELS, hidden=TCN_HIDDEN,
                 n_layers=TCN_LAYERS, dropout=TCN_DROPOUT):
        super().__init__()
        layers = []
        for i in range(n_layers):
            dilation = 2 ** i
            in_ch = in_channels if i == 0 else hidden
            layers.append(TCNBlock(in_ch, hidden, kernel_size=3,
                                   dilation=dilation, dropout=dropout))
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Linear(hidden, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        # x: (batch, channels, seq_len)
        h = self.tcn(x)          # (batch, hidden, seq_len)
        h = h.mean(dim=2)        # GlobalAvgPool -> (batch, hidden)
        return self.head(h).squeeze(-1)  # (batch,)


# Count parameters
model_test = TCN1D().to(DEVICE)
n_params = sum(p.numel() for p in model_test.parameters())
n_trainable = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'TCN1D architecture:')
print(f'  Input: ({N_CHANNELS}, {SEQ_LEN})')
print(f'  Hidden: {TCN_HIDDEN}, Layers: {TCN_LAYERS}')
print(f'  Dilation: {[2**i for i in range(TCN_LAYERS)]}')
print(f'  Receptive field: {sum(2**i * 2 for i in range(TCN_LAYERS)) + 1} frames')
print(f'  Total params: {n_params:,}')
print(f'  Trainable params: {n_trainable:,}')
print(f'\nModel structure:')
print(model_test)
del model_test

cell_end('Cell 5: Model Definition')


In [ ]:
# === Cell 6: LOSO Cross-Validation ===
cell_start('Cell 6: LOSO CV')


class GaitCycleDataset(Dataset):
    """Simple dataset wrapper for gait cycle sequences."""

    def __init__(self, X, y):
        self.X = torch.from_numpy(X)   # (N, 4, 64)
        self.y = torch.from_numpy(y).float()  # (N,)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def train_one_fold(X_train, y_train, X_val, y_val, pos_weight,
                   n_epochs=N_EPOCHS, patience=PATIENCE, verbose=False):
    """Train TCN for one LOSO fold.

    Returns:
        val_probs: (n_val,) predicted probabilities for validation set
        best_val_loss: float
    """
    model = TCN1D().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    pw = torch.tensor([pos_weight], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    train_ds = GaitCycleDataset(X_train, y_train)
    val_ds = GaitCycleDataset(X_val, y_val)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=torch.cuda.is_available())

    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(n_epochs):
        # Train
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validate
        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                vl = criterion(logits, yb).item()
                val_loss_sum += vl * len(xb)
                val_count += len(xb)

        val_loss = val_loss_sum / max(val_count, 1)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Load best model and predict
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    all_probs = []
    with torch.no_grad():
        for xb, _ in val_loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)

    val_probs = np.concatenate(all_probs) if all_probs else np.array([])
    return val_probs, best_val_loss


# ---------------------------------------------------------------
# LOSO CV loop
# ---------------------------------------------------------------

items_to_train = list(range(1, 18)) if not DRY_RUN else DRY_ITEMS

# Collect OOF predictions at patient-side level
oof_results = {}    # {item_i: {(pid, side): prob}}
tcn_loso_acc = {}   # {item_i: accuracy}
xgb_baseline_acc = {}  # For comparison
item_class_balance = {}

print(f'\n{"="*60}')
print(f'LOSO CV: {len(items_to_train)} items, '
      f'{len(train_pids_with_cycles)} patients with cycles')
print(f'{"="*60}\n')

_iter_times = []
_peak_vram = 0.0
_total_iters = 0

for item_i in items_to_train:
    item_start = time.time()

    # Build dataset for this item (with augmentation for training)
    X_all, y_all, pids_all = build_augmented_dataset(train_cycles, item_i, do_augment=True)

    # Also build unaugmented for validation
    X_noaug, y_noaug, pids_noaug = build_augmented_dataset(train_cycles, item_i, do_augment=False)

    if len(y_noaug) == 0:
        print(f'Item {item_i:2d}: No data, skipping')
        continue

    n_pos = int(y_noaug.sum())
    n_neg = len(y_noaug) - n_pos
    pos_weight = max(n_neg / max(n_pos, 1), 0.1)
    item_class_balance[item_i] = (n_pos, n_neg, pos_weight)

    # Determine folds
    unique_pids = sorted(set(pids_all))
    if DRY_RUN:
        unique_pids = unique_pids[:DRY_FOLDS]

    oof_item = {}

    pbar = tqdm(unique_pids, desc=f'Item {item_i:2d} LOSO', leave=False)
    for fold_pid in pbar:
        iter_start = time.time()

        # Split: train = all other patients (augmented), val = this patient (no augmentation)
        train_mask = pids_all != fold_pid
        val_mask = pids_noaug == fold_pid

        X_tr_fold = X_all[train_mask]
        y_tr_fold = y_all[train_mask]
        X_va_fold = X_noaug[val_mask]
        y_va_fold = y_noaug[val_mask]

        if len(X_va_fold) == 0 or len(X_tr_fold) == 0:
            continue

        val_probs, val_loss = train_one_fold(
            X_tr_fold, y_tr_fold, X_va_fold, y_va_fold,
            pos_weight=pos_weight
        )

        # Aggregate cycle-level probs to patient-side level
        val_cycles_for_pid = [c for c in train_cycles
                              if c['patient_id'] == fold_pid and c['labels'] is not None
                              and item_i in c['labels']]

        if len(val_probs) == len(val_cycles_for_pid):
            side_probs = defaultdict(list)
            for prob, cyc in zip(val_probs, val_cycles_for_pid):
                side_probs[cyc['side']].append(prob)

            for side, probs in side_probs.items():
                mean_prob = np.mean(probs)
                oof_item[(fold_pid, side)] = mean_prob

        iter_time = time.time() - iter_start
        _iter_times.append(iter_time)
        _total_iters += 1

        if _total_iters % 20 == 0 and torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1e9
            _peak_vram = max(_peak_vram, allocated)

        pbar.set_postfix(loss=f'{val_loss:.3f}',
                         vram=f'{_peak_vram:.1f}G' if torch.cuda.is_available() else 'CPU')

    # Compute LOSO accuracy for this item
    correct = 0
    total = 0
    for (pid, side), prob in oof_item.items():
        true_label = label_lookup.get((pid, side), {}).get(item_i)
        if true_label is not None:
            pred = 1 if prob >= 0.5 else 0
            correct += (pred == true_label)
            total += 1

    acc = correct / max(total, 1)
    tcn_loso_acc[item_i] = acc
    oof_results[item_i] = oof_item

    item_time = time.time() - item_start
    print(f'Item {item_i:2d}: TCN LOSO acc = {acc:.4f} '
          f'({correct}/{total}) | pos_weight={pos_weight:.2f} | '
          f'{item_time:.1f}s ({item_time/60:.1f}min)')

    log_gpu_stats(f'item_{item_i}')

# ---------------------------------------------------------------
# XGBoost baseline for comparison
# ---------------------------------------------------------------
print(f'\n{"="*60}')
print('Computing XGBoost LOSO baseline for comparison...')
print(f'{"="*60}')

try:
    from xgboost import XGBClassifier
    from sklearn.model_selection import LeaveOneGroupOut
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not available, using majority baseline')

FEAT_COLS = sorted([c for c in df_train.columns
                    if c.startswith(('sag_', 'cor_', 'all_', 'n_'))])
X_feat = np.nan_to_num(df_train[FEAT_COLS].values.astype(np.float32), nan=0.0)
groups = df_train['patient_id'].values

for item_i in tqdm(items_to_train, desc='XGB baseline'):
    y = df_train[f'evgs_{item_i}'].values.astype(int)
    if HAS_XGB:
        logo = LeaveOneGroupOut()
        oof = np.zeros_like(y)
        for tr_idx, te_idx in logo.split(X_feat, y, groups):
            n_p = y[tr_idx].sum()
            n_n = len(y[tr_idx]) - n_p
            spw = max(n_n / max(n_p, 1), 1.0)
            clf = XGBClassifier(
                n_estimators=200, max_depth=4, learning_rate=0.05,
                subsample=0.9, colsample_bytree=0.9,
                scale_pos_weight=spw,
                eval_metric='logloss', verbosity=0, n_jobs=2,
                use_label_encoder=False)
            clf.fit(X_feat[tr_idx], y[tr_idx])
            oof[te_idx] = clf.predict(X_feat[te_idx])
        xgb_baseline_acc[item_i] = (oof == y).mean()
    else:
        maj = int(y.mean() >= 0.5)
        xgb_baseline_acc[item_i] = max((y == 0).mean(), (y == 1).mean())

# ---------------------------------------------------------------
# Summary table
# ---------------------------------------------------------------
print(f'\n{"="*60}')
print(f'{"Item":>4} {"TCN":>8} {"XGB":>8} {"Delta":>8} {"Winner":>8} {"Pos%":>6}')
print(f'{"-"*46}')

items_tcn_wins = []
for item_i in items_to_train:
    tcn_a = tcn_loso_acc.get(item_i, 0)
    xgb_a = xgb_baseline_acc.get(item_i, 0)
    delta = tcn_a - xgb_a
    winner = 'TCN' if delta > 0.02 else ('XGB' if delta < -0.02 else 'TIE')
    n_pos, n_neg, pw = item_class_balance.get(item_i, (0, 0, 1.0))
    pos_pct = 100 * n_pos / max(n_pos + n_neg, 1)
    print(f'{item_i:>4d} {tcn_a:>8.4f} {xgb_a:>8.4f} {delta:>+8.4f} {winner:>8s} {pos_pct:>5.1f}%')
    if delta > 0.02:
        items_tcn_wins.append(item_i)

print(f'\nItems where TCN exceeds XGB OOF accuracy: {items_tcn_wins}')
mean_tcn = np.mean([tcn_loso_acc[i] for i in items_to_train if i in tcn_loso_acc])
mean_xgb = np.mean([xgb_baseline_acc[i] for i in items_to_train if i in xgb_baseline_acc])
print(f'Mean accuracy: TCN={mean_tcn:.4f}, XGB={mean_xgb:.4f}, delta={mean_tcn - mean_xgb:+.4f}')

cell_end('Cell 6: LOSO CV')


In [ ]:
# === Cell 7: Final Training + Test Predictions ===
cell_start('Cell 7: Final Training + Test Predictions')

test_predictions = {}  # {item_i: {(pid, side): prob}}

for item_i in tqdm(items_to_train, desc='Final training + test prediction'):
    # Build full training dataset (augmented)
    X_train_full, y_train_full, _ = build_augmented_dataset(
        train_cycles, item_i, do_augment=True)

    if len(X_train_full) == 0:
        print(f'  Item {item_i}: No training data, skipping')
        continue

    # Build test dataset (no augmentation)
    X_test_list = []
    test_cycle_info = []
    for cyc in test_cycles:
        X_test_list.append(cyc['angles'].T)  # (4, SEQ_LEN)
        test_cycle_info.append((cyc['patient_id'], cyc['side']))

    if not X_test_list:
        print(f'  Item {item_i}: No test cycles, skipping')
        continue

    X_test_arr = np.array(X_test_list, dtype=np.float32)
    y_test_dummy = np.zeros(len(X_test_arr), dtype=np.int64)

    # Class weight
    n_pos = int(y_train_full.sum())
    n_neg = len(y_train_full) - n_pos
    pos_weight = max(n_neg / max(n_pos, 1), 0.1)

    # Train model on all training data
    model = TCN1D().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    pw_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw_tensor)

    train_ds = GaitCycleDataset(X_train_full, y_train_full)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=torch.cuda.is_available())

    model.train()
    for epoch in range(N_EPOCHS):
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

    # Predict test cycles
    model.eval()
    test_ds = GaitCycleDataset(X_test_arr, y_test_dummy)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=0, pin_memory=torch.cuda.is_available())

    all_probs = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
    test_probs = np.concatenate(all_probs)

    # Aggregate per patient-side
    side_probs = defaultdict(list)
    for prob, (pid, side) in zip(test_probs, test_cycle_info):
        side_probs[(pid, side)].append(prob)

    item_preds = {}
    for (pid, side), probs in side_probs.items():
        item_preds[(pid, side)] = float(np.mean(probs))

    test_predictions[item_i] = item_preds

# Print test prediction summary
print(f'\nTest predictions generated for {len(test_predictions)} items:')
for item_i in sorted(test_predictions.keys()):
    preds = test_predictions[item_i]
    n_preds = len(preds)
    n_pos = sum(1 for p in preds.values() if p >= 0.5)
    print(f'  Item {item_i:2d}: {n_preds} patient-sides, '
          f'{n_pos} predicted abnormal ({100*n_pos/max(n_preds,1):.0f}%)')

cell_end('Cell 7: Final Training + Test Predictions')


In [ ]:
# === Cell 8: Generate Submissions ===
cell_start('Cell 8: Generate Submissions')

ITEM_COLS_L = [f'L{i}' for i in range(1, 18)]
ITEM_COLS_R = [f'R{i}' for i in range(1, 18)]
ITEM_COLS = ITEM_COLS_L + ITEM_COLS_R


def _preserve_track2_rows(df_new):
    """Preserve Track 2 rows from the base CSV (overwritten by notebook 03)."""
    # Track-2 rows are preserved from the current base CSV.
    return df_new


def recompute_total(df_new):
    """Recompute Total column from L1-L17, R1-R17."""
    t1 = df_new['ID'].str.startswith('track1-')
    df_new.loc[t1, 'Total'] = df_new.loc[t1, ITEM_COLS].sum(axis=1).astype(int)
    return df_new


def make_tcn_csv(items_to_replace, suffix, description):
    """Generate a submission CSV replacing specified items with TCN predictions."""
    if df_base is not None:
        df_new = df_base.copy()
    else:
        # Build from scratch if no base
        rows = []
        for pid in T1_TEST_IDS:
            row = {'ID': f'track1-{pid}'}
            for c in ITEM_COLS:
                row[c] = 0
            row['Total'] = 0
            row['Left_gait_subtype'] = -1
            row['Right_gait_subtype'] = -1
            rows.append(row)
        for pid in T2_TEST_IDS:
            row = {'ID': f'track2-{pid}'}
            for c in ITEM_COLS:
                row[c] = 0
            row['Total'] = 0
            row['Left_gait_subtype'] = 'WNL'
            row['Right_gait_subtype'] = 'WNL'
            rows.append(row)
        df_new = pd.DataFrame(rows)

    n_changes = 0
    n_kept = 0
    change_details = []

    for pid in T1_TEST_IDS:
        rid = f'track1-{pid}'
        row_mask = df_new['ID'] == rid
        if not row_mask.any():
            continue

        for side in ('left', 'right'):
            side_letter = 'L' if side == 'left' else 'R'
            for item_i in items_to_replace:
                col = f'{side_letter}{item_i}'
                preds = test_predictions.get(item_i, {})
                prob = preds.get((pid, side))

                if prob is None:
                    n_kept += 1
                    continue

                new_val = 1 if prob >= 0.5 else 0
                old_val = int(df_new.loc[row_mask, col].values[0])

                if old_val != new_val:
                    df_new.loc[row_mask, col] = new_val
                    n_changes += 1
                    change_details.append(
                        f'  P{pid} {side_letter}{item_i}: {old_val}->{new_val} (p={prob:.3f})')

    df_new = _preserve_track2_rows(df_new)
    df_new = recompute_total(df_new)

    out_path = f'{DIAGNOSTIC_DIR}/tcn_item14_{suffix}.csv'  # candidate variants for diagnostic review only
    df_new.to_csv(out_path, index=False)
    return out_path, n_changes, n_kept, description, change_details


generated = []

# 1. Per-item CSVs
for item_i in items_to_train:
    if item_i not in test_predictions:
        continue
    suffix = f'item{item_i:02d}'
    desc = f'Item {item_i} only, TCN acc={tcn_loso_acc.get(item_i, 0):.4f}'
    result = make_tcn_csv([item_i], suffix, desc)
    generated.append(result)

# 2. TCN-preferred combo (items where TCN OOF accuracy exceeds XGB)
if items_tcn_wins:
    suffix = 'tcn_wins_combo'
    desc = f'Items where TCN exceeds XGB OOF accuracy: {items_tcn_wins}'
    result = make_tcn_csv(items_tcn_wins, suffix, desc)
    generated.append(result)

# 3. All items combo
suffix = 'all_items'
desc = f'All {len(items_to_train)} items replaced by TCN'
result = make_tcn_csv(items_to_train, suffix, desc)
generated.append(result)

# 4. Top-N items by TCN OOF accuracy
sorted_items = sorted(items_to_train, key=lambda i: tcn_loso_acc.get(i, 0), reverse=True)
for n_top in [3, 5]:
    if len(sorted_items) >= n_top:
        top_items = sorted_items[:n_top]
        suffix = f'top{n_top}_tcn'
        desc = f'Top-{n_top} TCN items: {top_items}'
        result = make_tcn_csv(top_items, suffix, desc)
        generated.append(result)

# 5. High-accuracy subset: items where TCN LOSO accuracy meets a threshold
high_acc_items = [i for i in items_to_train if tcn_loso_acc.get(i, 0) >= 0.85]
if high_acc_items:
    suffix = 'high_acc_combo'
    desc = f'Items with TCN acc >= 0.85: {high_acc_items}'
    result = make_tcn_csv(high_acc_items, suffix, desc)
    generated.append(result)

print(f'\n{"="*60}')
print(f'Generated {len(generated)} diagnostic candidate CSVs (canonical L14/R14 source written by Cell 8b):')
print(f'{"="*60}')

for p, desc, ch, kp, details in [(r[0], r[3], r[1], r[2], r[4]) for r in generated]:
    print(f'\n  {os.path.basename(p)}')
    print(f'    changes={ch}, fell_back={kp}')
    print(f'    {desc}')
    if details and ch > 0:
        for d in details[:10]:
            print(f'    {d}')
        if len(details) > 10:
            print(f'    ... and {len(details)-10} more changes')

cell_end('Cell 8: Generate Submissions')


In [ ]:
# === Cell 8b: Emit canonical item-14 artifact ===
# Selection rule (codified in this notebook):
#   The canonical Track-1 source for EVGS item 14 is the per-item-14 TCN
#   prediction CSV. Final assembly only consumes the L14 / R14 columns
#   from this file; no other candidate (tcn_wins_combo, top-N, all_items,
#   high_acc_combo) is used. Picking the per-item CSV keeps item 14's
#   final-CSV contribution independent of how other items perform in the
#   same combo, which matches the provenance of the per-item-14 source slice released as the canonical artifact.
#
# This cell expects Cell 8 to have written {OUTPUT_DIR}/tcn_item14_item14.csv.
# It fails fast if missing rather than reading the committed item-14 slice,
# ensuring the canonical runtime artifact is the freshly trained TCN output.
# __canonical_item14_emit__
import os
import shutil

per_item14_path = f'{DIAGNOSTIC_DIR}/tcn_item14_item14.csv'  # written by Cell 8 into diagnostics/
canonical_path  = f'{OUTPUT_DIR}/item14_tcn.csv'

if not os.path.exists(per_item14_path):
    raise FileNotFoundError(
        f'Canonical item-14 source missing: {per_item14_path}. '
        'Cell 8 was expected to produce it. Re-run Cell 8 (or this notebook end-to-end) '
        'before running final assembly.')

shutil.copyfile(per_item14_path, canonical_path)
print(f'Canonical item-14 artifact written: {canonical_path}')
print(f'  source: {per_item14_path}')
print(f'  Final assembly will consume L14/R14 from this file.')

In [ ]:
# === Cell 9: Summary ===
cell_start('Summary')

total_time = sum(t for t in _cell_times.values() if t is not None)
time_report = '\n'.join(
    f'  {name}: {t:.1f}s ({t/60:.1f}min)'
    for name, t in _cell_times.items() if t is not None
)

# Per-item results
item_lines = []
for item_i in items_to_train:
    tcn_a = tcn_loso_acc.get(item_i, 0)
    xgb_a = xgb_baseline_acc.get(item_i, 0)
    delta = tcn_a - xgb_a
    winner = 'TCN' if delta > 0.02 else ('XGB' if delta < -0.02 else 'TIE')
    item_lines.append(f'  Item {item_i:2d}: TCN={tcn_a:.4f}  XGB={xgb_a:.4f}  '
                      f'delta={delta:+.4f}  {winner}')
item_report = '\n'.join(item_lines)

# CSV list
csv_lines = '\n'.join(
    f'  {os.path.basename(r[0])}  (changes={r[1]})'
    for r in generated
) if generated else '  (none)'

# GPU stats
if _iter_times:
    avg_iter = sum(_iter_times) / len(_iter_times)
    throughput_info = f'{avg_iter:.3f}s/fold'
else:
    avg_iter = 0
    throughput_info = 'N/A'

# Auto suggestions
suggestions = []
if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_util = _peak_vram / total_vram * 100 if total_vram > 0 else 0
    if vram_util < 50:
        suggestions.append(f'VRAM only {vram_util:.0f}% used. Model is very lightweight.')
    suggestions.append(f'Peak VRAM: {_peak_vram:.1f} / {total_vram:.1f} GB ({vram_util:.0f}%)')
else:
    suggestions.append('Running on CPU. GPU would significantly speed up training.')

if items_tcn_wins:
    suggestions.append(f'Items where TCN exceeds XGB OOF accuracy: {items_tcn_wins}.')
else:
    suggestions.append('TCN OOF accuracy is comparable to XGB on all items. '
                       'Augmentation, model depth, and ensembling with XGB are tunable knobs.')

auto_suggestion = '\n  '.join(suggestions)

mean_tcn_all = np.mean([tcn_loso_acc.get(i, 0) for i in items_to_train]) if items_to_train else 0
mean_xgb_all = np.mean([xgb_baseline_acc.get(i, 0) for i in items_to_train]) if items_to_train else 0

_n_params_summary = sum(p.numel() for p in TCN1D().parameters())

print(f"""
{'='*60}
SUMMARY - {NOTEBOOK_NAME}
{'='*60}

Execution time:
{time_report}
  -----
  Total: {total_time:.1f}s ({total_time/60:.1f}min)

Environment:
  GPU: {gpu_name}
  VRAM: {vram_gb:.0f} GB
  Batch size: {BATCH_SIZE}
  DRY_RUN: {DRY_RUN}
  Device: {DEVICE}

Data:
  Train cycles: {len(train_cycles)} from {len(train_pids_with_cycles)} patients
  Test cycles: {len(test_cycles)} from {len(test_pids_with_cycles)} patients
  Augmentation: {AUG_FACTOR}x
  Sequence length: {SEQ_LEN} frames, {N_CHANNELS} channels

Model:
  TCN hidden={TCN_HIDDEN}, layers={TCN_LAYERS}, dropout={TCN_DROPOUT}
  LR={LR}, weight_decay={WEIGHT_DECAY}, epochs={N_EPOCHS}
  Params per model: {_n_params_summary:,}

Per-item LOSO accuracy (TCN vs XGB):
{item_report}

  Mean: TCN={mean_tcn_all:.4f}, XGB={mean_xgb_all:.4f}, delta={mean_tcn_all - mean_xgb_all:+.4f}
  Items where TCN exceeds XGB OOF accuracy: {items_tcn_wins if items_tcn_wins else 'none'}

Training stats:
  Avg time per fold: {throughput_info}
  Total folds trained: {_total_iters}

Generated CSVs ({len(generated)}):
{csv_lines}

Optimization suggestions:
  {auto_suggestion}

Generated CSV artifacts for offline review:

Notes:
  - Track 2 values set to the clinical baseline
  - Base submission: {base_path_used if base_path_used else 'built from scratch'}
  - If DRY_RUN = os.environ.get('CV4CHL_DRY_RUN', '0') == '1'  # set CV4CHL_DRY_RUN=1 for ~2-min smoke test, set to False and rerun for full results
{'='*60}
""")

cell_end('Summary')
